In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import math
import warnings
from sklearn.model_selection import KFold
from torch.utils.data import Subset
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
SEQ_LEN = 64          # Temporal window size for the Transformer
BATCH_SIZE = 32
D_MODEL = 128         # Transformer embedding dimension
N_HEADS = 8
NUM_LAYERS = 2
EPOCHS = 20        # Increased from 5
PATIENCE = 3       # How many epochs to wait before giving up
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. DATA PREPARATION & FEATURE FUSION
# ==========================================
def clean_and_prepare_data(filepath, is_train=True, scaler=None):
    """Loads, cleans string artifacts, and engineers grouped features."""
    df = pd.read_csv(filepath)
    
    # Define features based on the problem statement
    feature_cols = [
        'Carrier_Doppler_hz', 'Carrier_phase', # Carrier Dynamics
        'EC', 'LC', 'PC',                             # Correlator Shape
        'PIP', 'PQP',                                 # Quality Metrics
        'TCD', 'TOW',              # Timing
        'CN0', 'Pseudorange_m'                        # Standalone & Base
    ]
    
    # THE FIX: Force PRN to numeric first. Garbage strings become NaN.
    df['PRN'] = pd.to_numeric(df['PRN'], errors='coerce')
    
    # 1. Clean Garbage Rows (Apply numeric conversion to the rest)
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop rows where ANY of our crucial features OR the PRN turned into NaN
    df = df.dropna(subset=feature_cols + ['PRN'])
    
    # 2. Feature Engineering (Grouping & Relationships)
    df['EML_Shape'] = df['EC'] - df['LC'] # Early-Minus-Late discriminator
    df['Signal_Valid_Mask'] = (df['PRN'] > 0).astype(float) # Now this works perfectly!
    
    all_features = feature_cols + ['EML_Shape', 'Signal_Valid_Mask']
    
    # 2. Feature Engineering (Grouping & Relationships)
    df['EML_Shape'] = df['EC'] - df['LC'] 
    df['Signal_Valid_Mask'] = (df['PRN'] > 0).astype(float) 
    
    # --- NEW: PAPER-INSPIRED CNO FEATURES ---
    
    # Feature A: Cross-Channel Spatial Variance (Are all signals coming from one source?)
    # Calculate the variance of CNO across all active channels for each specific timestamp
    # We replace 0s with NaN temporarily so empty channels don't skew the variance
    df['CNO_masked'] = df['CN0'].where(df['PRN'] > 0, np.nan)
    spatial_variance = df.groupby('time')['CNO_masked'].transform('std').fillna(0)
    df['CNO_Spatial_Std'] = spatial_variance
    
    # Feature B: Temporal Variance (Is the signal abnormally stable over time?)
    # Calculate a rolling standard deviation for each specific PRN (Satellite)
    df = df.sort_values(by=['PRN', 'time'])
    df['CNO_Temporal_Std'] = df.groupby('PRN')['CNO_masked'].transform(lambda x: x.rolling(window=5, min_periods=1).std()).fillna(0)
    
    # Add these to your feature list!
    all_features = feature_cols + ['EML_Shape', 'Signal_Valid_Mask', 'CNO_Spatial_Std', 'CNO_Temporal_Std']
    # 3. Scaling (Fit on train, transform test)
    if is_train:
        scaler = StandardScaler()
        df[all_features] = scaler.fit_transform(df[all_features])
    else:
        df[all_features] = scaler.transform(df[all_features])
        
    # 4. Early Fusion: Reshape to (Timesteps, 8_Channels, Features)
    df = df.sort_values(by=['time', 'channel'])
    
    grouped = df.groupby('time')
    time_indices = list(grouped.groups.keys())
    
    num_times = len(time_indices)
    num_channels = 8 
    num_features = len(all_features)
    
    X = np.zeros((num_times, num_channels, num_features))
    y = np.zeros((num_times,))
    
    for i, (time_val, group) in enumerate(grouped):
        X[i] = group[all_features].values
        if is_train:
            y[i] = group['spoofed'].iloc[0] 
            
    X_fused = X.reshape(num_times, num_channels * num_features)
    
    return X_fused, y, time_indices, scaler

# ==========================================
# 3. PYTORCH DATASETS
# ==========================================
class GNSSSequenceDataset(Dataset):
    def __init__(self, X, y=None, seq_len=64):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
        self.seq_len = seq_len
        
        # Calculate how many full sequences we can make
        self.num_sequences = math.ceil(len(X) / seq_len)

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start_idx = idx * self.seq_len
        end_idx = min((idx + 1) * self.seq_len, len(self.X))
        
        seq_x = self.X[start_idx:end_idx]
        
        # Pad sequence if it's the last incomplete chunk
        if len(seq_x) < self.seq_len:
            pad_len = self.seq_len - len(seq_x)
            seq_x = torch.cat([seq_x, torch.zeros(pad_len, seq_x.shape[1])])
            
        if self.y is not None:
            seq_y = self.y[start_idx:end_idx]
            if len(seq_y) < self.seq_len:
                seq_y = torch.cat([seq_y, torch.zeros(pad_len)])
            return seq_x, seq_y.unsqueeze(-1)
        
        return seq_x

# ==========================================
# 4. TRANSFORMER ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class GNSS_Spoof_Detector(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layer calculates "confidence of spoofing in that time step"
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (batch, seq_len, 8_channels * features)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        out = self.classifier(x)
        return out # shape: (batch, seq_len, 1)

# ==========================================
# 5. TRAINING & INFERENCE PIPELINE
# ==========================================
def main():
    print("Loading and preparing data...")
    X_train_full, y_train_full, train_times, scaler = clean_and_prepare_data('/kaggle/input/datasets/vooooooom/signals/train.csv', is_train=True)
    X_test, _, test_times, _ = clean_and_prepare_data('/kaggle/input/datasets/vooooooom/signals/test.csv', is_train=False, scaler=scaler)
    
    input_dim = X_train_full.shape[1] 
    
    # --- NEW: Create the full sequence dataset FIRST ---
    full_train_ds = GNSSSequenceDataset(X_train_full, y_train_full, seq_len=SEQ_LEN)
    
    # Prepare the test loader
    test_ds = GNSSSequenceDataset(X_test, seq_len=SEQ_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # Array to accumulate test predictions for ensembling
    ensembled_test_preds = np.zeros(len(test_times))
    
    N_SPLITS = 5
    # --- NEW: Shuffled K-Fold to ensure class balance across folds ---
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    
    print(f"Starting {N_SPLITS}-Fold Shuffled Cross-Validation...")
    
    # kf.split() will now split the INDICES of our sequences, not raw rows
    for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(full_train_ds)))):
        print(f"\n{'='*20} Fold {fold+1}/{N_SPLITS} {'='*20}")
        print(f"Train sequences: {len(train_idx)} | Val sequences: {len(val_idx)}")
        
        # --- NEW: Use PyTorch Subset to grab the sequences for this fold ---
        train_ds = Subset(full_train_ds, train_idx)
        val_ds = Subset(full_train_ds, val_idx)
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        model = GNSS_Spoof_Detector(input_dim, D_MODEL, N_HEADS, NUM_LAYERS).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.BCELoss() 
        
        best_val_f1 = -1.0
        best_model_path = f'best_spoof_detector_fold_{fold+1}.pth'
        epochs_no_improve = 0  # <--- NEW: Track patience
        
        for epoch in range(EPOCHS):
            # ---------- TRAINING PHASE ----------
            model.train()
            train_loss = 0
            train_preds, train_targets = [], []
            
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                
                optimizer.zero_grad()
                predictions = model(batch_x)
                loss = criterion(predictions, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                
                preds_binary = (predictions > 0.5).float().cpu().numpy().flatten()
                targets = batch_y.cpu().numpy().flatten()
                train_preds.extend(preds_binary)
                train_targets.extend(targets)
                
            train_f1 = f1_score(train_targets, train_preds, average='weighted')
            
            # ---------- VALIDATION PHASE ----------
            model.eval()
            val_loss = 0
            val_preds, val_targets = [], []
            
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                    
                    predictions = model(batch_x)
                    loss = criterion(predictions, batch_y)
                    val_loss += loss.item()
                    
                    preds_binary = (predictions > 0.5).float().cpu().numpy().flatten()
                    targets = batch_y.cpu().numpy().flatten()
                    val_preds.extend(preds_binary)
                    val_targets.extend(targets)
                    
            val_f1 = f1_score(val_targets, val_preds, average='weighted')
            
            print(f"Epoch {epoch+1}/{EPOCHS} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
            
# --- UPDATED: Save model OR trigger early stopping ---
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), best_model_path)
                print(f"  --> Best model for fold {fold+1} saved! (Val F1: {best_val_f1:.4f})")
                epochs_no_improve = 0  # Reset counter if we improved!
            else:
                epochs_no_improve += 1
                print(f"  --> No improvement for {epochs_no_improve} epoch(s).")
                
                if epochs_no_improve >= PATIENCE:
                    print(f"  --> Early stopping triggered! Moving to next fold.")
                    break  # Break out of the epoch loop early
                
        # ---------- FOLD INFERENCE ----------
        print(f"Loading best model for Fold {fold+1} to predict on Test Set...")
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        
        fold_confidences = []
        with torch.no_grad():
            for batch_x in test_loader:
                batch_x = batch_x.to(DEVICE)
                predictions = model(batch_x).cpu().numpy().squeeze(-1)
                
                for seq_preds in predictions:
                    fold_confidences.extend(seq_preds)
                    
        fold_confidences = fold_confidences[:len(test_times)]
        ensembled_test_preds += np.array(fold_confidences)

    # ==========================================
    # ENSEMBLING & SUBMISSION
    # ==========================================
    print("\nGenerating Final Ensembled Submission...")
    ensembled_test_preds /= N_SPLITS
    
    submission_df = pd.DataFrame({
        'time': test_times,
        'Confidence': ensembled_test_preds
    })
    submission_df['Spoofed'] = (submission_df['Confidence'] > 0.5).astype(int) 
    
    submission_df = submission_df[['time', 'Spoofed', 'Confidence']]
    submission_df.to_csv('submission_final.csv', index=False)
    print("Ensembled submission saved to 'submission_final.csv'!")

if __name__ == "__main__":
    main()

Loading and preparing data...
Starting 5-Fold Shuffled Cross-Validation...

==================== Fold 1/5 ====================
Train sequences: 1392 | Val sequences: 349
Epoch 1/20 | Train F1: 0.9047 | Val F1: 0.9710
  --> Best model for fold 1 saved! (Val F1: 0.9710)
Epoch 2/20 | Train F1: 0.9579 | Val F1: 0.9711
  --> Best model for fold 1 saved! (Val F1: 0.9711)
Epoch 3/20 | Train F1: 0.9637 | Val F1: 0.9687
  --> No improvement for 1 epoch(s).
Epoch 4/20 | Train F1: 0.9626 | Val F1: 0.9702
  --> No improvement for 2 epoch(s).
Epoch 5/20 | Train F1: 0.9627 | Val F1: 0.9769
  --> Best model for fold 1 saved! (Val F1: 0.9769)
Epoch 6/20 | Train F1: 0.9678 | Val F1: 0.9758
  --> No improvement for 1 epoch(s).
Epoch 7/20 | Train F1: 0.9675 | Val F1: 0.9776
  --> Best model for fold 1 saved! (Val F1: 0.9776)
Epoch 8/20 | Train F1: 0.9697 | Val F1: 0.9767
  --> No improvement for 1 epoch(s).
Epoch 9/20 | Train F1: 0.9700 | Val F1: 0.9767
  --> No improvement for 2 epoch(s).
Epoch 10/20 | 